# Movie Graph Demo

**Pipeline:** Synthetic data → Parquet → GCS → FalkorDB → Cypher

This notebook loads a **Movie Graph** using Method 2 (direct Parquet upload — no Starburst needed).

Graph shape:
- **Movie** nodes: `movie_id`, `title`, `year`, `genre`
- **Actor** nodes: `actor_id`, `name`, `birth_year`
- **Director** nodes: `director_id`, `name`
- **ACTED_IN** edges: Actor → Movie
- **DIRECTED** edges: Director → Movie

In [ ]:
# ── Step 1 │ Setup & Imports ───────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 1 │ Setup & Imports ───────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
import requests, json, time, os, subprocess
import pandas as pd
from pathlib import Path

API = "http://graph-olap-control-plane:8080"
H   = {"X-Username": "demo@example.com", "X-User-Role": "admin", "Content-Type": "application/json"}

def get(path):       r = requests.get(f"{API}{path}", headers=H);  r.raise_for_status(); return r.json()
def post(path, body): r = requests.post(f"{API}{path}", headers=H, json=body); r.raise_for_status(); return r.json()
def delete(path):    r = requests.delete(f"{API}{path}", headers=H); r.raise_for_status(); return r.json()

print(requests.get(f"{API}/health").json())

## Step 1 — Generate synthetic movie data

In [ ]:
# ── Step 2 │ Generate Synthetic Data ───────────────────────────────
# Services : local Python — no network
# ──────────────────────────────────────────────────────────────────────
# ── Step 2 │ Generate Synthetic Data ──────────────────────────────────────
# Services : (local Python — no network)
# ──────────────────────────────────────────────────────────────────────────
# --- Movies ---
movies = pd.DataFrame([
    {"movie_id": 1,  "title": "The Matrix",          "year": 1999, "genre": "Sci-Fi"},
    {"movie_id": 2,  "title": "John Wick",            "year": 2014, "genre": "Action"},
    {"movie_id": 3,  "title": "Speed",                "year": 1994, "genre": "Action"},
    {"movie_id": 4,  "title": "Interstellar",          "year": 2014, "genre": "Sci-Fi"},
    {"movie_id": 5,  "title": "The Dark Knight",       "year": 2008, "genre": "Action"},
    {"movie_id": 6,  "title": "Inception",             "year": 2010, "genre": "Sci-Fi"},
    {"movie_id": 7,  "title": "Tenet",                 "year": 2020, "genre": "Sci-Fi"},
    {"movie_id": 8,  "title": "Dunkirk",               "year": 2017, "genre": "War"},
    {"movie_id": 9,  "title": "The Matrix Reloaded",   "year": 2003, "genre": "Sci-Fi"},
    {"movie_id": 10, "title": "Point Break",           "year": 1991, "genre": "Action"},
])

# --- Actors ---
actors = pd.DataFrame([
    {"actor_id": 1, "name": "Keanu Reeves",     "birth_year": 1964},
    {"actor_id": 2, "name": "Carrie-Anne Moss",  "birth_year": 1967},
    {"actor_id": 3, "name": "Laurence Fishburne","birth_year": 1961},
    {"actor_id": 4, "name": "Matthew McConaughey","birth_year": 1969},
    {"actor_id": 5, "name": "Anne Hathaway",     "birth_year": 1982},
    {"actor_id": 6, "name": "Leonardo DiCaprio", "birth_year": 1974},
    {"actor_id": 7, "name": "Tom Hardy",         "birth_year": 1977},
    {"actor_id": 8, "name": "Cillian Murphy",    "birth_year": 1976},
    {"actor_id": 9, "name": "Mark Rylance",      "birth_year": 1960},
])

# --- Directors ---
directors = pd.DataFrame([
    {"director_id": 1, "name": "The Wachowskis"},
    {"director_id": 2, "name": "Chad Stahelski"},
    {"director_id": 3, "name": "Jan de Bont"},
    {"director_id": 4, "name": "Christopher Nolan"},
    {"director_id": 5, "name": "Kathryn Bigelow"},
])

# --- ACTED_IN edges ---
acted_in = pd.DataFrame([
    {"actor_id": 1, "movie_id": 1},   # Keanu → The Matrix
    {"actor_id": 2, "movie_id": 1},   # Carrie-Anne → The Matrix
    {"actor_id": 3, "movie_id": 1},   # Fishburne → The Matrix
    {"actor_id": 1, "movie_id": 9},   # Keanu → Matrix Reloaded
    {"actor_id": 2, "movie_id": 9},
    {"actor_id": 3, "movie_id": 9},
    {"actor_id": 1, "movie_id": 2},   # Keanu → John Wick
    {"actor_id": 1, "movie_id": 3},   # Keanu → Speed
    {"actor_id": 1, "movie_id": 10},  # Keanu → Point Break
    {"actor_id": 4, "movie_id": 4},   # McConaughey → Interstellar
    {"actor_id": 5, "movie_id": 4},   # Hathaway → Interstellar
    {"actor_id": 6, "movie_id": 6},   # DiCaprio → Inception
    {"actor_id": 7, "movie_id": 6},   # Hardy → Inception
    {"actor_id": 7, "movie_id": 5},   # Hardy → Dark Knight
    {"actor_id": 8, "movie_id": 5},   # Murphy → Dark Knight
    {"actor_id": 7, "movie_id": 7},   # Hardy → Tenet
    {"actor_id": 8, "movie_id": 8},   # Murphy → Dunkirk
    {"actor_id": 9, "movie_id": 8},   # Rylance → Dunkirk
])

# --- DIRECTED edges ---
directed = pd.DataFrame([
    {"director_id": 1, "movie_id": 1},
    {"director_id": 1, "movie_id": 9},
    {"director_id": 2, "movie_id": 2},
    {"director_id": 3, "movie_id": 3},
    {"director_id": 4, "movie_id": 4},
    {"director_id": 4, "movie_id": 5},
    {"director_id": 4, "movie_id": 6},
    {"director_id": 4, "movie_id": 7},
    {"director_id": 4, "movie_id": 8},
    {"director_id": 5, "movie_id": 10},
])

print(f"Movies: {len(movies)}, Actors: {len(actors)}, Directors: {len(directors)}")
print(f"ACTED_IN edges: {len(acted_in)}, DIRECTED edges: {len(directed)}")

## Step 2 — Save to Parquet

In [ ]:
# ── Step 3 │ Save to Parquet ───────────────────────────────────────
# Services : local filesystem — no network
# ──────────────────────────────────────────────────────────────────────
# ── Step 3 │ Save to Parquet ───────────────────────────────────────────────
# Services : (local filesystem — no network)
# ──────────────────────────────────────────────────────────────────────────
base = Path("/tmp/movie-graph")
for p in ["nodes/Movie", "nodes/Actor", "nodes/Director", "edges/ACTED_IN", "edges/DIRECTED"]:
    (base / p).mkdir(parents=True, exist_ok=True)

movies.to_parquet(base / "nodes/Movie/data.parquet",       index=False)
actors.to_parquet(base / "nodes/Actor/data.parquet",       index=False)
directors.to_parquet(base / "nodes/Director/data.parquet", index=False)
acted_in.to_parquet(base / "edges/ACTED_IN/data.parquet",  index=False)
directed.to_parquet(base / "edges/DIRECTED/data.parquet",  index=False)

print("✅ Parquet files written to /tmp/movie-graph/")

## Step 3 — Create Mapping

In [ ]:
# ── Step 4 │ Create Mapping ────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 4 │ Create Mapping ────────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
mapping = post("/api/mappings", {
    "name": "Movie Graph",
    "description": "Movies, actors, directors — synthetic dataset",
    "node_definitions": [
        {
            "label": "Movie",
            "sql": "SELECT movie_id, title, year, genre FROM placeholder",
            "primary_key": {"name": "movie_id", "type": "INT64"},
            "properties": [
                {"name": "title", "type": "STRING"},
                {"name": "year",  "type": "INT64"},
                {"name": "genre", "type": "STRING"},
            ]
        },
        {
            "label": "Actor",
            "sql": "SELECT actor_id, name, birth_year FROM placeholder",
            "primary_key": {"name": "actor_id", "type": "INT64"},
            "properties": [
                {"name": "name",       "type": "STRING"},
                {"name": "birth_year", "type": "INT64"},
            ]
        },
        {
            "label": "Director",
            "sql": "SELECT director_id, name FROM placeholder",
            "primary_key": {"name": "director_id", "type": "INT64"},
            "properties": [{"name": "name", "type": "STRING"}]
        },
    ],
    "edge_definitions": [
        {
            "type": "ACTED_IN",
            "from_node": "Actor",
            "to_node": "Movie",
            "sql": "SELECT actor_id, movie_id FROM placeholder",
            "from_key": "actor_id",
            "to_key": "movie_id",
            "properties": []
        },
        {
            "type": "DIRECTED",
            "from_node": "Director",
            "to_node": "Movie",
            "sql": "SELECT director_id, movie_id FROM placeholder",
            "from_key": "director_id",
            "to_key": "movie_id",
            "properties": []
        },
    ]
})

mapping_id = mapping["data"]["id"]
print(f"✅ Mapping created: id={mapping_id}")

## Step 4 — Create Instance

In [ ]:
# ── Step 5 │ Create Instance + Bypass Export Worker ────────────────
# Services : Control Plane (graph-olap-control-plane:8080)  ·  PostgreSQL (postgres:5432)
# ──────────────────────────────────────────────────────────────────────
# ── Step 5 │ Create Instance + Bypass Export Worker ───────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)  ·  PostgreSQL (postgres:5432)
# ──────────────────────────────────────────────────────────────────────────
instance = post("/api/instances", {
    "mapping_id":  mapping_id,
    "wrapper_type": "falkordb",
    "name":        "Movie Graph Instance",
    "ttl":         "PT4H",
})
instance_id = instance["data"]["id"]
snapshot_id = instance["data"]["snapshot_id"]
print(f"Instance created: id={instance_id}")
print(f"Snapshot id:      {snapshot_id}")

# Immediately fail any pending export_jobs and mark the snapshot ready.
# This bypasses the export worker (no Starburst needed for local demo).
import psycopg2
conn = psycopg2.connect(
    host="postgres", port=5432,
    dbname="control_plane", user="control_plane", password="control_plane"
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute(
        "UPDATE export_jobs SET status='failed' WHERE snapshot_id=%s AND status='pending'",
        (snapshot_id,)
    )
    print(f"  Failed {cur.rowcount} pending export_job(s)")
    cur.execute(
        "UPDATE snapshots SET status='ready' WHERE id=%s",
        (snapshot_id,)
    )
    print(f"  Snapshot {snapshot_id} marked ready")
conn.close()

## Step 5 — Upload Parquet files to GCS

In [ ]:
# ── Step 6 │ Upload Parquet to GCS ─────────────────────────────────
# Services : fake-gcs-local (fake-gcs-local:4443)
# ──────────────────────────────────────────────────────────────────────
# ── Step 6 │ Upload Parquet to GCS ────────────────────────────────────────
# Services : fake-gcs-local (fake-gcs-local:4443)
# ──────────────────────────────────────────────────────────────────────────
# Upload parquet files directly to fake-gcs via HTTP API
import requests as _req

BUCKET = "graph-olap-local-dev"
GCS = "http://fake-gcs-local:4443"
# GCS path must match what the wrapper expects
OWNER  = "demo@example.com"
PREFIX = f"{OWNER}/{mapping_id}/v1/{snapshot_id}"
print(f"Uploading to gs://{BUCKET}/{PREFIX}/")

# Ensure bucket exists in fake-gcs (in-memory — recreated after each pod restart)
_br = _req.post(f"{GCS}/storage/v1/b", json={"name": BUCKET},
               headers={"Content-Type": "application/json"})
if _br.status_code not in (200, 201, 409):
    _br.raise_for_status()

files = [
    ("nodes/Movie/data.parquet",       f"{PREFIX}/nodes/Movie/data.parquet"),
    ("nodes/Actor/data.parquet",       f"{PREFIX}/nodes/Actor/data.parquet"),
    ("nodes/Director/data.parquet",    f"{PREFIX}/nodes/Director/data.parquet"),
    ("edges/ACTED_IN/data.parquet",    f"{PREFIX}/edges/ACTED_IN/data.parquet"),
    ("edges/DIRECTED/data.parquet",    f"{PREFIX}/edges/DIRECTED/data.parquet"),
]

for local, remote in files:
    file_path = base / local
    with open(file_path, "rb") as f:
        data = f.read()
    url = f"{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={remote}"
    resp = _req.post(url, data=data, headers={"Content-Type": "application/octet-stream"})
    if resp.status_code in (200, 201):
        print(f"  ✅ {remote}")
    else:
        print(f"  ❌ {remote}: {resp.status_code} {resp.text[:200]}")


## Step 6 — Mark snapshot as ready

## Step 7 — Wait for instance to be running

In [ ]:
# ── Step 7 │ Wait for Instance ─────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 7 │ Wait for Instance ────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
print("Polling... (reconciliation loop runs every ~30s)")
start = time.time()
while True:
    elapsed = int(time.time() - start)
    inst    = get(f"/api/instances/{instance_id}")
    status  = inst["data"]["status"]
    print(f"  [{elapsed:3d}s] instance={status}")
    if status == "running":
        break
    if status in ("stopped", "failed"):
        raise RuntimeError(f"Instance ended with status: {status}")
    time.sleep(10)

print(f"\n🎉 Movie graph is running! FalkorDB pod ready.")

## Step 8 — Query the Movie Graph

In [ ]:
# ── Step 8 │ Connect & Query Graph ─────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8 │ Connect & Query Graph ────────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
inst_data = get(f"/api/instances/{instance_id}")["data"]
pod_name  = inst_data["pod_name"]
WRAPPER   = f"http://{pod_name}:8000"
print(f"Wrapper: {WRAPPER}")

def cypher(q, params=None):
    body = {"query": q}
    if params:
        body["parameters"] = params
    for attempt in range(10):
        try:
            r = requests.post(f"{WRAPPER}/query",
                              headers={"Content-Type": "application/json"}, json=body)
            r.raise_for_status()
            return r.json()["rows"]
        except Exception as e:
            if attempt < 9:
                print(f"  Wrapper not ready yet, retrying in 3s... ({attempt+1}/10)")
                time.sleep(3)
            else:
                raise

# --- Node counts ---
print("=== Graph loaded — Node counts ===")
for row in cypher("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC"):
    print(f"  {str(row[0]):20s}: {row[1]:,}")

In [ ]:
# ── Step 8a │ Cypher — Nolan Filmography ───────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8a │ Cypher — Nolan Filmography ──────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Movies by Nolan ---
print("=== Christopher Nolan filmography ===")
results = cypher("""
    MATCH (d:Director {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie)
    RETURN m.title AS title, m.year AS year, m.genre AS genre
    ORDER BY m.year
""")
for row in results:
    print(f"  {row[1]}  {row[0]}  ({row[2]})")

In [ ]:
# ── Step 8b │ Cypher — Actors in 2+ Nolan Films ────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8b │ Cypher — Multi-film Actors ──────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Actors who appeared in multiple Nolan films ---
print("=== Actors in 2+ Nolan films ===")
results = cypher("""
    MATCH (d:Director {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Actor)
    WITH a, count(m) AS movies, collect(m.title) AS titles
    WHERE movies >= 2
    RETURN a.name AS actor, movies, titles
    ORDER BY movies DESC
""")
for row in results:
    print(f"  {row[0]:25s}  {row[1]} films: {row[2]}")

In [ ]:
# ── Step 8c │ Cypher — Actor Filmography ───────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8c │ Cypher — Actor Filmography ──────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- How many films has Keanu been in? ---
print("=== Keanu Reeves filmography ===")
results = cypher("""
    MATCH (a:Actor {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie)
    RETURN m.title AS title, m.year AS year
    ORDER BY m.year
""")
for row in results:
    print(f"  {row[1]}  {row[0]}")

In [ ]:
# ── Step 8d │ Cypher — Co-star Pairs ───────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8d │ Cypher — Co-star Pairs ──────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# --- Actors who share movies (co-starred) ---
print("=== Co-star pairs ===")
results = cypher("""
    MATCH (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
    WHERE a1.actor_id < a2.actor_id
    RETURN a1.name AS actor1, a2.name AS actor2, m.title AS movie
    ORDER BY a1.name
""")
for row in results:
    print(f"  {row[0]:25s} + {row[1]:25s}  in '{row[2]}'")

In [ ]:
# ── Step 8e │ Cypher — Actor Connection Path ───────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────
# ── Step 8e │ Cypher — Actor Connection Path ──────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)
# ──────────────────────────────────────────────────────────────────────────
# FalkorDB does not support undirected shortestPath.
# Instead, find the connecting actor/movie chain manually.
print("=== Connection: Keanu Reeves → Cillian Murphy ===")
results = cypher("""
    MATCH (a:Actor {name: 'Keanu Reeves'})-[:ACTED_IN]->(m1:Movie)
    MATCH (b:Actor {name: 'Cillian Murphy'})-[:ACTED_IN]->(m2:Movie)
    MATCH (bridge:Actor)-[:ACTED_IN]->(m1)
    MATCH (bridge)-[:ACTED_IN]->(m2)
    WHERE bridge.name <> 'Keanu Reeves' AND bridge.name <> 'Cillian Murphy'
    RETURN a.name AS from, m1.title AS via_movie1, bridge.name AS bridge,
           m2.title AS via_movie2, b.name AS to
    LIMIT 5
""")
if results:
    for row in results:
        print(f"  {row[0]} → {row[1]} ← {row[2]} → {row[3]} ← {row[4]}")
else:
    print("  No 2-hop connection found — they share no common co-stars in this dataset")


## Step 9 — Visualise (PyVis)

In [ ]:
# ── Step 9 │ Visualise Graph (PyVis) ───────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)  ·  PyVis (local)
# ──────────────────────────────────────────────────────────────────────
# ── Step 9 │ Visualise Graph (PyVis) ──────────────────────────────────────
# Services : FalkorDB Wrapper ({pod_name}:8000)  ·  PyVis (local)
# ──────────────────────────────────────────────────────────────────────────
from pyvis.network import Network
from IPython.display import IFrame

net = Network(height="600px", width="100%", bgcolor="#1a1a2e", font_color="white", notebook=True)
net.toggle_physics(True)

# Color scheme
colors = {"Movie": "#e94560", "Actor": "#0f3460", "Director": "#533483"}

# Add all nodes and edges
rows = cypher("""
    MATCH (a:Actor)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Director)
    RETURN a.actor_id AS a_id, a.name AS a_name,
           m.movie_id AS m_id, m.title AS m_title,
           d.director_id AS d_id, d.name AS d_name
""")

for row in rows:
    a_id, a_name, m_id, m_title, d_id, d_name = row
    net.add_node(f"a_{a_id}", label=a_name,  color=colors["Actor"],    size=20, title=f"Actor: {a_name}")
    net.add_node(f"m_{m_id}", label=m_title, color=colors["Movie"],    size=30, title=f"Movie: {m_title}", shape="star")
    net.add_node(f"d_{d_id}", label=d_name,  color=colors["Director"], size=25, title=f"Director: {d_name}")
    net.add_edge(f"a_{a_id}", f"m_{m_id}", title="ACTED_IN",  color="#888")
    net.add_edge(f"d_{d_id}", f"m_{m_id}", title="DIRECTED",  color="#ff9900", width=2)

net.show("movie_graph.html")
IFrame("movie_graph.html", width="100%", height=620)

## Cleanup (optional)

In [ ]:
# ── Step 10 │ Cleanup ──────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────
# ── Step 10 │ Cleanup ─────────────────────────────────────────────────────
# Services : Control Plane (graph-olap-control-plane:8080)
# ──────────────────────────────────────────────────────────────────────────
# Clean up — delete the instance to free the wrapper pod memory
try:
    delete(f"/api/instances/{instance_id}")
    print(f"Instance {instance_id} deleted — wrapper pod will be removed")
except Exception as e:
    print(f"Instance {instance_id} already gone or not found: {e}")